# Lakehouse Pipeline - Exploratory Analysis

A quick look at the tables the pipeline produces. Everything here reads from
Postgres: the `marts` schema built by dbt and the `analytics` schema written by
the Spark jobs.

Run the pipeline first (see the README), then run this notebook.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

engine = create_engine(
    "postgresql+psycopg2://%s:%s@%s:%s/%s" % (
        os.environ.get("POSTGRES_USER", "postgres"),
        os.environ.get("POSTGRES_PASSWORD", "postgres"),
        os.environ.get("POSTGRES_HOST", "localhost"),
        os.environ.get("POSTGRES_PORT", "5432"),
        os.environ.get("POSTGRES_DB", "ecommerce"),
    )
)

print("Connected")

## Orders

`marts.fct_orders` is one row per order with the line items rolled up.

In [ ]:
orders = pd.read_sql("SELECT * FROM marts.fct_orders", engine)

print("%d orders loaded" % len(orders))
orders.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(orders["net_amount"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Order value distribution")
axes[0].set_xlabel("Net amount ($)")

daily = orders[~orders["is_cancelled"]].groupby("order_date")["net_amount"].sum()
axes[1].plot(daily.index, daily.values, color="steelblue")
axes[1].set_title("Daily revenue")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Where the money comes from

Revenue split by payment method and by shipping country.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

by_payment = orders.groupby("payment_method")["net_amount"].sum().sort_values()
by_payment.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Revenue by payment method")
axes[0].set_xlabel("Revenue ($)")

by_country = orders.groupby("shipping_country")["net_amount"].sum().nlargest(10).sort_values()
by_country.plot(kind="barh", ax=axes[1], color="seagreen")
axes[1].set_title("Top 10 countries by revenue")
axes[1].set_xlabel("Revenue ($)")

plt.tight_layout()
plt.show()

## Conversion funnel

`marts.mart_conversion_funnel` has one row per day with the drop-off between
each step of the funnel.

In [ ]:
funnel = pd.read_sql(
    "SELECT * FROM marts.mart_conversion_funnel ORDER BY date DESC LIMIT 30",
    engine,
)

latest = funnel.iloc[0]
print("Latest day: %s" % latest["date"])
print("Conversion rate: %.2f%%" % (latest["overall_conversion_rate"] * 100))

steps = {
    "Sessions": latest["total_sessions"],
    "Added to cart": latest["sessions_with_cart"],
    "Started checkout": latest["sessions_with_checkout"],
    "Purchased": latest["sessions_with_purchase"],
}

plt.figure(figsize=(8, 4))
plt.barh(list(steps.keys())[::-1], list(steps.values())[::-1], color="steelblue")
plt.title("Funnel on %s" % latest["date"])
plt.xlabel("Sessions")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(funnel["date"], funnel["overall_conversion_rate"], marker="o")
plt.title("Conversion rate over time")
plt.ylabel("Conversion rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Cohort retention

Of the users who first ordered in a given month, how many came back in the
months after.

In [ ]:
cohort = pd.read_sql("SELECT * FROM marts.mart_cohort_retention", engine)

pivot = cohort.pivot(
    index="cohort_month",
    columns="months_since_first_order",
    values="retention_rate",
)
pivot.index = pivot.index.astype(str)

plt.figure(figsize=(12, 6))
sns.heatmap(pivot, annot=True, fmt=".0%", cmap="Blues", vmin=0, vmax=1, linewidths=0.5)
plt.title("Monthly cohort retention")
plt.xlabel("Months since first order")
plt.ylabel("Cohort")
plt.tight_layout()
plt.show()

## User segments

RFM segments come from the Spark `user_features` job.

In [ ]:
users = pd.read_sql(
    "SELECT rfm_segment, ltv_estimate, order_count_180d, total_spend_180d "
    "FROM marts.dim_users WHERE rfm_segment IS NOT NULL",
    engine,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

users["rfm_segment"].value_counts().plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Users per RFM segment")

users.groupby("rfm_segment")["ltv_estimate"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="seagreen"
)
axes[1].set_title("Average estimated LTV by segment")
axes[1].set_xlabel("LTV ($)")

plt.tight_layout()
plt.show()

## Products

`marts.dim_products` joins the catalogue with the last 30 days of engagement
from the streaming side of the pipeline.

In [ ]:
products = pd.read_sql("SELECT * FROM marts.dim_products", engine)

top = products.nlargest(15, "views_30d").sort_values("views_30d")

plt.figure(figsize=(10, 6))
plt.barh(top["product_name"], top["views_30d"], color="steelblue")
plt.title("Most viewed products (last 30 days)")
plt.xlabel("Views")
plt.tight_layout()
plt.show()

In [ ]:
# Is a product that gets a lot of views actually converting?
viewed = products[products["views_30d"] > 0]

plt.figure(figsize=(8, 5))
plt.scatter(viewed["views_30d"], viewed["avg_conversion_rate_30d"], alpha=0.6)
plt.title("Views vs conversion rate")
plt.xlabel("Views (30d)")
plt.ylabel("Average conversion rate")
plt.tight_layout()
plt.show()